# Ep_ISA_NEW rerun notebook: GitHub clone -> preflight audit -> ISA -> checks

This notebook reruns the updated Ep_ISA_NEW workflow from a GitHub repository. It does not draw Nature figures.

Order:
1. Clone the latest Ep_ISA_NEW code from GitHub and install it editable.
2. Load models, promoter regions and FASTA.
3. Build Fi-NeMo motif_locs for CAGE, DEV and HK.
4. Run preflight motif single/pair and overlap audit first.
5. Clean old ISA outputs, then rerun ISA with updated deepISA-style logic.
6. Run strict post-run audits before any plotting.

Do not mix these outputs with the old `Ep_ISA` / `results_cage` / `results_dev` / `results_hk` outputs.


## 0. Runtime setup from GitHub

First push the whole local `Ep_ISA_NEW` folder to GitHub. Then set `REPO_URL` below to that repository.

This cell always removes `/content/Ep_ISA_NEW` and clones a fresh copy, so Colab cannot silently reuse stale code.


In [ ]:
!pip install tensorflow tf-keras bioframe pyBigWig loguru statsmodels h5py matplotlib-venn -q

import os, sys, json, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from loguru import logger

from google.colab import drive
drive.mount('/content/drive')

# EDIT THESE TWO LINES after pushing Ep_ISA_NEW to GitHub.
REPO_URL = 'https://github.com/YOUR_NAME/Ep_ISA_NEW.git'
REPO_BRANCH = 'main'
assert 'YOUR_NAME' not in REPO_URL, 'Set REPO_URL to your real GitHub repository first.'

LOCAL_REPO = Path('/content/Ep_ISA_NEW')
if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(LOCAL_REPO)
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(LOCAL_REPO), '-q'], check=True)

from Ep_ISA_NEW.quickstart import EpQuickStart
from Ep_ISA_NEW.scoring.preflight import run_preflight_pair_audit
from Ep_ISA_NEW.scoring.audit import audit_task_results
from Ep_ISA_NEW.scoring.utils_isa import region_str_to_seq

print('Using Ep_ISA_NEW from:', LOCAL_REPO)
print('GitHub repo:', REPO_URL, 'branch:', REPO_BRANCH)


## 1. Paths and configuration

In [ ]:
BASE_DIR    = Path('/content/drive/MyDrive/DeepEpromote/Drosophila')
IC_TRIM_DIR = BASE_DIR / 'Motif_cluster/ic_trimmed_results'
SCAN_DIR    = IC_TRIM_DIR / 'finemo_validation_scans/stage2_core_promoter/finemo_scans'
FINEMO_H5   = IC_TRIM_DIR / 'IC_Trimmed_MetaClusters_for_Finemo.h5'
PROM_FILE   = BASE_DIR / 'DeepCAGE/DATA/starr_cage_reconstructed.tsv'
CAGE_MODEL  = BASE_DIR / 'DeepCAGE/model/model_starr_ctss_T1/best_model.h5'
STARR_JSON  = BASE_DIR / 'DeepSTARR/model_artifacts/DeepSTARR.model.json'
STARR_H5    = BASE_DIR / 'DeepSTARR/model_artifacts/DeepSTARR.model.h5'

# New output root. This keeps old results_cage/results_dev/results_hk untouched.
OUT_ROOT = IC_TRIM_DIR / 'ep_isa_new_rerun_results'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

TASKS = {
    'CAGE': {'scan': 'CAGE_NEW', 'results': OUT_ROOT / 'results_cage_newisa', 'track': 0, 'model': 'cage'},
    'DEV':  {'scan': 'DEV',      'results': OUT_ROOT / 'results_dev_newisa',  'track': 0, 'model': 'starr'},
    'HK':   {'scan': 'HK',       'results': OUT_ROOT / 'results_hk_newisa',   'track': 1, 'model': 'starr'},
}

ISA_CONFIG_BASE = {
    'null_percentile': 80,
    'min_count': 10,
    'q_val_thresh': 0.1,
    'receptive_field': 255,
    'single_null_n_samples': 8192,
    'pair_null_n_samples': 8192,
    'pair_null_n_bins': 20,
    'tau_quantile': 50.0,
    'single_filter_mode': 'positive_all_tracks',
    'num_regions_per_batch': 200,
    'pred_batch_size': 1024,
}

for p in [SCAN_DIR, FINEMO_H5, PROM_FILE, CAGE_MODEL, STARR_JSON, STARR_H5]:
    assert Path(p).exists(), f'Missing path: {p}'
print('Output root:', OUT_ROOT)

## 2. Load models

In [ ]:
cage_model = tf.keras.models.load_model(
    str(CAGE_MODEL), custom_objects={'mse': tf.keras.losses.MeanSquaredError()})
print(f'CAGE model: input={cage_model.input_shape}, output={cage_model.output_shape}')

import tf_keras
with open(STARR_JSON, 'r') as f:
    src = tf_keras.models.model_from_json(f.read())
src.load_weights(str(STARR_H5))
tmp = '/content/_tmp_starr_ep_isa_new.h5'
src.save(tmp)
starr_model = tf.keras.models.load_model(tmp, compile=False)
os.remove(tmp)
print(f'DeepSTARR model: input={starr_model.input_shape}, output={starr_model.output_shape}  [0]=DEV [1]=HK')

## 3. Build promoter regions and FASTA

In [ ]:
prom = pd.read_csv(PROM_FILE, sep='\t').rename(columns={
    'Chromosome': 'chrom', 'Start': 'start', 'End': 'end',
    'CAGE_dom_strand': 'strand', 'CAGE_dom_log2TPM': 'y', 'Sequence': 'seq'})
prom['sc'] = 'mismatch'
prom.loc[prom['refseq_tss_strand'] == prom['strand'], 'sc'] = 'match'
prom.loc[prom[['refseq_tss_strand', 'strand']].isna().any(axis=1), 'sc'] = 'NA'
prom_19777 = prom[prom.sc == 'match'].reset_index(drop=True)
assert len(prom_19777) == 19777

df_regions = prom_19777[['chrom', 'start', 'end']].copy()
df_regions['peak_id'] = df_regions.index
df_regions = df_regions[['peak_id', 'chrom', 'start', 'end']]

FASTA_PATH = Path('/content/prom_regions_dm3_ep_isa_new.fa')
with open(FASTA_PATH, 'w') as f:
    for _, r in prom_19777.iterrows():
        f.write(f">{r['chrom']}:{int(r['start'])}-{int(r['end'])}\n{r['seq']}\n")

print('regions:', len(df_regions))
print('FASTA:', FASTA_PATH)

# Sanity check: Ep_ISA_NEW supports this region-level FASTA directly.
import bioframe as bf
from Ep_ISA_NEW.scoring.utils_isa import region_str_to_seq
fasta_check = bf.load_fasta(str(FASTA_PATH))
seq_lens = [len(region_str_to_seq(fasta_check, f"{r.chrom}:{int(r.start)}-{int(r.end)}")) for _, r in df_regions.head(5).iterrows()]
print('First 5 FASTA lengths via Ep_ISA_NEW:', seq_lens)
assert len(set(seq_lens)) == 1, seq_lens

## 3b. Model IO and prediction-matrix audit

This checks the matrix contract between notebook and scripts before ISA: sequence length, model input shape, model output shape, requested tracks and returned prediction matrix shape.


In [ ]:
from Ep_ISA_NEW.modeling.predict import audit_model_io, compute_predictions

audit_regions = [f"{r.chrom}:{int(r.start)}-{int(r.end)}" for _, r in df_regions.head(8).iterrows()]
audit_seqs = [region_str_to_seq(fasta_check, r) for r in audit_regions]
print("Audit sequence lengths:", sorted({len(s) for s in audit_seqs}))

io_audits = []
for task_name, model_obj, tracks in [
    ('CAGE', cage_model, [0]),
    ('DEV', starr_model, [0]),
    ('HK', starr_model, [1]),
]:
    row = audit_model_io(model_obj, audit_seqs, tracks=tracks)
    row['task'] = task_name
    io_audits.append(row)
    assert row["prediction_shape"] == (len(audit_seqs), len(tracks)), row

io_audit_df = pd.DataFrame(io_audits)
display(io_audit_df)
io_audit_df.to_csv(OUT_ROOT / "ep_isa_new_model_io_matrix_audit.csv", index=False)

# Negative control: CAGE is a single-output model; requesting track 1 must fail loudly.
try:
    compute_predictions(cage_model, audit_seqs[:2], tracks=[1], batch_size=2)
    raise AssertionError("CAGE track 1 unexpectedly succeeded; track validation is broken.")
except ValueError as e:
    print('Expected CAGE track validation error:', e)


## 4. Create task objects and Fi-NeMo motif_locs

This step writes `motif_locs.csv` and `non_motif_locs.csv`, but does not run ISA yet.

In [ ]:
MODEL_BY_NAME = {'cage': cage_model, 'starr': starr_model}
qs = {}

for task, cfg in TASKS.items():
    results_dir = Path(cfg['results'])
    results_dir.mkdir(parents=True, exist_ok=True)
    q = EpQuickStart(results_dir=str(results_dir), fasta_path=str(FASTA_PATH), df_regions=df_regions)
    q.define_model(MODEL_BY_NAME[cfg['model']])
    q.load_finemo(
        hits_tsv_path=str(SCAN_DIR / cfg['scan'] / 'hits.tsv'),
        finemo_h5_path=str(FINEMO_H5),
        auto_threshold_percentile=50,
    )
    qs[task] = q
    print(task, 'motif_locs ->', q.files['motif_locs'])

## 5. Preflight audit first

Run this before any ISA scoring. Check whether single motif loci already exceed valid non-overlapping motif pairs, and inspect skipped overlap/abutting pairs.

In [ ]:
preflight_rows = []
for task, q in qs.items():
    cfg = TASKS[task]
    summary = run_preflight_pair_audit(
        motif_locs_path=q.files['motif_locs'],
        out_summary_path=q.files['preflight_pair_audit'],
        out_region_path=q.files['preflight_pair_audit_by_region'],
        out_overlap_path=q.files['preflight_overlap_pairs'],
        receptive_field=ISA_CONFIG_BASE['receptive_field'],
    )
    row = summary.iloc[0].to_dict()
    row['task'] = task
    preflight_rows.append(row)

preflight = pd.DataFrame(preflight_rows).set_index('task')
display(preflight[[
    'motif_locs_rows_raw', 'motif_locs_rows_after_new_dedup',
    'all_pairs_same_region', 'receptive_field_pairs',
    'pair_to_single_ratio_receptive_field',
    'overlapping_or_adjacent_pairs', 'overlap_exact_same_tf_pairs',
    'overlap_different_family_pairs', 'too_far_pairs'
]])

In [ ]:
# Top skipped overlap/abutting TF pairs. These are not standard ISA motif pairs.
for task, q in qs.items():
    p = Path(q.files['preflight_overlap_pairs'])
    if not p.exists() or p.stat().st_size == 0:
        print(task, 'no overlap/abutting pairs')
        continue
    df = pd.read_csv(p)
    if df.empty:
        print(task, 'no overlap/abutting pairs')
        continue
    pair = df.apply(lambda r: '|'.join(sorted([str(r.tf1), str(r.tf2)])), axis=1)
    print('\n' + task)
    display(pair.value_counts().head(15).rename('n').to_frame())

## 5b. Count ledger: raw motifs -> candidate pairs -> ISA outputs

This table is the main count check. Before full ISA, it shows raw Fi-NeMo hits, motif_locs after Ep_ISA_NEW loading/thresholding/deduplication, and candidate motif pairs under the deepISA pair rule. After full ISA, rerun this cell with `include_algorithm_outputs=True` to add `motif_single_isa.csv` and `motif_combi_isa.csv` counts.

In [ ]:
def build_count_ledger(include_algorithm_outputs=False):
    rows = []
    for task, q in qs.items():
        cfg = TASKS[task]
        scan_hits = SCAN_DIR / cfg['scan'] / 'hits.tsv'
        motif_locs = Path(q.files['motif_locs'])
        preflight_path = Path(q.files['preflight_pair_audit'])

        raw_hits_n = len(pd.read_csv(scan_hits, sep='\t'))
        motif_locs_n = len(pd.read_csv(motif_locs)) if motif_locs.exists() else np.nan
        pf = pd.read_csv(preflight_path).iloc[0] if preflight_path.exists() else None

        row = {
            'task': task,
            'raw_finemo_hits_tsv': raw_hits_n,
            'motif_locs_after_load_finemo': motif_locs_n,
            'motif_locs_after_new_dedup': int(pf.motif_locs_rows_after_new_dedup) if pf is not None else np.nan,
            'all_same_region_candidate_pairs': int(pf.all_pairs_same_region) if pf is not None else np.nan,
            'valid_nonoverlap_rf_candidate_pairs': int(pf.receptive_field_pairs) if pf is not None else np.nan,
            'skipped_overlap_or_abutting_pairs': int(pf.overlapping_or_adjacent_pairs) if pf is not None else np.nan,
            'skipped_too_far_pairs': int(pf.too_far_pairs) if pf is not None else np.nan,
        }

        if include_algorithm_outputs:
            data = Path(q.data_dir)
            single_path = data / 'motif_single_isa.csv'
            combi_path = data / 'motif_combi_isa.csv'
            row['single_motif_after_single_isa_filter'] = len(pd.read_csv(single_path)) if single_path.exists() else np.nan
            row['motif_pairs_after_combi_isa'] = len(pd.read_csv(combi_path)) if combi_path.exists() else np.nan
            if combi_path.exists():
                combi = pd.read_csv(combi_path)
                inter = f"interaction_t{cfg['track']}"
                row['motif_pairs_with_non_nan_new_interaction'] = int(combi[inter].notna().sum()) if inter in combi.columns else np.nan
                row['motif_pairs_with_nan_new_interaction'] = int(combi[inter].isna().sum()) if inter in combi.columns else np.nan

        rows.append(row)
    return pd.DataFrame(rows)

count_ledger_pre_isa = build_count_ledger(include_algorithm_outputs=False)
display(count_ledger_pre_isa)
count_ledger_pre_isa.to_csv(OUT_ROOT / 'ep_isa_new_count_ledger_pre_isa.csv', index=False)

## 6. Rerun ISA with Ep_ISA_NEW

This is the long model-scoring step. It writes updated single ISA, combi ISA, null tables, normalized interaction and coop summaries.

Use this only after the preflight and count ledger look reasonable. When rerunning after code/filter changes, keep `CLEAN_OLD_ISA_OUTPUTS=True`; otherwise stale CSVs can contaminate checks.

Set `RUN_FULL_ISA = True` to start the full rerun.


In [ ]:
RUN_FULL_ISA = False  # inspect preflight first, then change to True
CLEAN_OLD_ISA_OUTPUTS = True  # keep True for every rerun after code/filter changes

if not RUN_FULL_ISA:
    print('Full ISA rerun is disabled. Set RUN_FULL_ISA=True after checking preflight tables.')
else:
    for task, q in qs.items():
        cfg = dict(ISA_CONFIG_BASE)
        cfg['tracks'] = [TASKS[task]['track']]
        if CLEAN_OLD_ISA_OUTPUTS:
            for key in ['null_isa', 'pred_orig', 'isa_single', 'isa_combi', 'null_interaction', 'imp_tf', 'coop_tf_pair', 'coop_tf']:
                p = Path(q.files[key])
                if p.exists():
                    p.unlink()
            for p in Path(q.data_dir).glob('coop_tf*_t*.csv'):
                p.unlink()
        print('\n=== Running', task, '===')
        q.run_isa(isa_config=cfg, start_from='single_isa')
        print(task, 'done ->', q.results_dir)

## 7. Post-run checks

Run after the full ISA rerun. Checks are designed for updated deepISA-style normalized interaction.

In [ ]:
count_ledger_post_isa = build_count_ledger(include_algorithm_outputs=True)
display(count_ledger_post_isa)

post_count_path = OUT_ROOT / 'ep_isa_new_count_ledger_post_isa.csv'
count_ledger_post_isa.to_csv(post_count_path, index=False)
print('Saved:', post_count_path)

In [ ]:
checks = []
fail_reason_tables = []
for task, q in qs.items():
    summary, fail_reasons = audit_task_results(
        data_dir=q.data_dir,
        track=TASKS[task]['track'],
        null_percentile=ISA_CONFIG_BASE['null_percentile'],
        expected_null_n=ISA_CONFIG_BASE['single_null_n_samples'],
    )
    summary['task'] = task
    summary['audit_pass'] = bool(
        summary['null_isa_near_expected']
        and summary['null_interaction_near_expected']
        and summary['pred_orig_has_target_track']
        and summary['single_positive_filter_consistent']
        and summary['interaction_gate_matches_non_nan']
    )
    checks.append(summary)
    fail_reasons.insert(0, 'task', task)
    fail_reason_tables.append(fail_reasons)

checks = pd.DataFrame(checks)
fail_reasons = pd.concat(fail_reason_tables, ignore_index=True)
display(checks)
display(fail_reasons)

check_path = OUT_ROOT / 'ep_isa_new_postrun_checks.csv'
fail_path = OUT_ROOT / 'ep_isa_new_interaction_gate_fail_reasons.csv'
checks.to_csv(check_path, index=False)
fail_reasons.to_csv(fail_path, index=False)
print('Saved:', check_path)
print('Saved:', fail_path)

## 8. Output handoff to plotting

Use these result folders as input for a new plotting notebook only after checks pass:

- CAGE: `results_cage_newisa`
- DEV: `results_dev_newisa`
- HK: `results_hk_newisa`

Do not mix these updated ISA outputs with the old `results_cage/results_dev/results_hk` outputs.